# Gider Talebi Analizi

Bu not defteri, yerel makbuz görüntülerinden seyahat giderlerini işlemek, bir gider talebi e-postası oluşturmak ve gider verilerini pasta grafikle görselleştirmek için eklentiler kullanan ajanların nasıl oluşturulacağını göstermektedir. Ajanlar, görev bağlamına göre dinamik olarak işlevleri seçer.

Adımlar:
1. OCR Ajanı yerel makbuz görüntüsünü işler ve seyahat gideri verilerini çıkarır.
2. E-posta Ajanı gider talebi e-postası oluşturur.

### Bir seyahat gideri senaryosu örneği:
Diyelim ki başka bir şehirdeki bir iş toplantısı için seyahat eden bir çalışansınız. Şirketiniz, tüm makul seyahatle ilgili giderleri karşılamayı taahhüt eder. İşte olası seyahat giderlerinin dökümü:
- Ulaşım:
Ev şehrinizden varış şehrine gidiş-dönüş uçak bileti.
Havalimanına gidiş-dönüş için taksi veya yolcu taşımacılığı hizmetleri.
Varış şehrinde yerel ulaşım (toplu taşıma, kiralık arabalar veya taksiler gibi).

- Konaklama:
Toplantı mekanına yakın orta sınıf bir iş otelinde üç gece konaklama.

- Yemekler:
Şirketin günlük ödenek politikasına göre kahvaltı, öğle ve akşam yemekleri için günlük yemek ödeneği.

- Çeşitli Giderler:
Havalimanı otopark ücretleri.
Otelde internet erişim ücretleri.
Bahşiş veya küçük hizmet ücretleri.

- Belgeler:
Geri ödeme için tüm makbuzları (uçuşlar, taksiler, otel, yemekler vb.) ve tamamlanmış gider raporunu teslim edersiniz.


## Gerekli kütüphaneleri içe aktar

Defter için gerekli kütüphaneleri ve modülleri içe aktarın.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import Content, Message, tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Gider Modellerini Tanımla

 Bireysel giderler için bir Pydantic modeli oluşturun ve bir kullanıcı sorgusunu yapılandırılmış gider verilerine dönüştürmek için bir ExpenseFormatter sınıfı oluşturun.

 Her gider şu formatta temsil edilecektir:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Araçların Tanımlanması - E-postanın Oluşturulması

Bir masraf talebi göndermek için e-posta oluşturacak bir araç fonksiyonu oluşturun.
- Bu araç, Microsoft Agent Framework'ten `@tool` dekoratörünü kullanır.
- Giderlerin toplam tutarını hesaplar ve detayları e-posta gövdesi olarak biçimler.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Seyahat Gider Makbuz Görüntülerini Yükleme

Makbuz görüntülerini yerel multimodal içerik olarak yüklemek için bir yardımcı fonksiyon oluşturun.
- Makbuz görüntü baytlarını okur ve bunları ajan analiz edebilmesi için Agent Framework `Content` içinde sarar.


In [ ]:
def load_receipt_image(image_path: str = "receipt.jpg") -> Content:
    """Load a receipt image as native multimodal content."""
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    return Content.from_data(image_bytes, "image/jpeg")

## İşlem Giderleri

Temsilcileri tanımlayın ve onları ardışık bir iş akışına bağlayın `WorkflowBuilder` kullanarak.
- OCR temsilcisi, kullanıcı mesajında verilen fiş görüntüsünden yapılandırılmış gider verilerini çıkarır.
- Email temsilcisi, çıkarılan verileri alır ve `generate_expense_email` aracı kullanarak profesyonel bir gider talep e-postası oluşturur.
- `WorkflowBuilder`, `add_edge` ile ardışık bir boru hattı oluşturur: OCR Temsilcisi → Email Temsilcisi.


In [ ]:
ocr_agent = client.as_agent(
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Analyze the receipt image supplied in the user message and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Ana fonksiyon

Ardışık iş akışını oluşturun ve makbuz görüntüsünü işlemek ve gider talep e-postası oluşturmak için çalıştırın.


> **Not:** Makbuz baytları yerel Ajan Çerçevesi multimodal `Content` içinde sarılmıştır, bu yüzden Foundry onları sıradan metin yerine görüntü girişi olarak alır.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
receipt_message = Message(
    role="user",
    contents=[prompt, load_receipt_image("receipt.jpg")],
)

events = workflow.run(
    receipt_message,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
